# Module 2 Lab: Count & Classify
## 6.371x — Probability and Statistical Data Analysis

### Where We Left Off

In Module 1, you made histograms of the FL/CL ratio and asked whether the distribution might contain hidden subgroups. You chose bin counts for visual clarity and computed empirical proportions.

### What's Changed

Two things are different now:

1. **You have more data.** We've collected measurements from 100 crabs total (50 new ones added to your original 50). More data means more resolution.

2. **You have new information.** Each crab is now labeled as **Male (M)** or **Female (F)**. This wasn't available before.

And there may be more about these crabs that we haven't told you yet.

### What's New Conceptually

In Module 1, your histograms were **pictures**. In this lab, they become **probability models**. You'll learn that:

- Choosing bins defines a **sample space**
- Normalizing counts gives you an **empirical PMF**
- You can compute **expectation** and **variance** from a PMF
- Splitting by sex gives you **conditional PMFs**
- A contingency table encodes a **joint distribution**
- You can start asking whether variables are **independent**


In [ ]:
# ── Setup: Data Loading for Colab ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    
    base_url = "https://raw.githubusercontent.com/codey-m/prob_stats/main/"
    url = base_url + filename
    print(f"📥 Loading from GitHub: {filename}")
    return pd.read_csv(url)

## Part 1: Return to the Ratio

Let's load the expanded dataset and recompute the ratio.


In [ ]:
crabs = load_data("crabs_module2.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))
crabs.head()

Shape: (100, 3)
Columns: ['sex', 'FL', 'CL']


,sex,FL,CL
0,F,15.6,31.2
1,F,13.4,28.4
2,M,19.8,42.4
3,M,13.2,27.1
4,F,18.0,34.7


### TODO 1a

Compute the ratio column, just like Module 1.


In [4]:
# TODO 1a: Compute the ratio
crabs["ratio"] = ...  # FL divided by CL

print(f"Number of crabs: {len(crabs)}")
print(f"Mean ratio:  {crabs['ratio'].mean():.4f}")
print(f"Std dev:     {crabs['ratio'].std():.4f}")

Number of crabs: 100


TypeError: unsupported operand type(s) for +: 'ellipsis' and 'ellipsis'

### TODO 1b

How do these summary statistics compare to Module 1 (where you had 50 crabs)? Are they similar? Should they be?

We'll ignore the `sex` column for now — we'll come back to it.


*Your answer here*

## Part 2: Discretization — Bins as a Sample Space

In Module 1, you chose bin counts for visual clarity. Now we're going to think about bins differently.

When you discretize a continuous measurement into bins, you are **defining a sample space**. If you create 5 bins, your sample space is $$\Omega = \{1, 2, 3, 4, 5\}$$. Each crab maps to exactly one outcome. This is a modeling choice — and it has consequences.

Let's start with a prescribed choice: **5 equal-width bins** spanning the range of the ratio.


In [5]:
# Prescribed: 5 equal-width bins
n_bins = 5
bin_edges = np.linspace(crabs["ratio"].min(), 
                        crabs["ratio"].max(), 
                        n_bins + 1)

print("Bin edges:")
for i in range(n_bins):
    print(f"  Bin {i+1}: [{bin_edges[i]:.4f}, {bin_edges[i+1]:.4f})")

# Assign each crab to a bin
# pd.cut returns a categorical variable
crabs["bin"] = pd.cut(crabs["ratio"], bins=bin_edges, 
                       labels=range(1, n_bins + 1),
                       include_lowest=True)

# Count how many crabs in each bin
counts = crabs["bin"].value_counts().sort_index()
print("\nCounts per bin:")
print(counts)

TypeError: '<=' not supported between instances of 'ellipsis' and 'ellipsis'

### Visualize: Module 1 Style vs Module 2 Style

Let's plot the same data two ways. On the left, the Module 1 histogram (raw counts, many bins, continuous feel). On the right, our 5-bin discretization (a bar chart of a discrete sample space).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Module 1 style — histogram with many bins
axes[0].hist(crabs["ratio"], bins=20, color="seagreen", 
             edgecolor="white", alpha=0.8)
axes[0].set_xlabel("FL / CL Ratio")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Module 1: Histogram (20 bins)")

# Right: Module 2 style — bar chart of discrete counts
bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 
               for i in range(n_bins)]
bar_width = (bin_edges[1] - bin_edges[0]) * 0.8

axes[1].bar(bin_centers, counts.values, width=bar_width,
            color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Ratio Bin (center)")
axes[1].set_ylabel("Count")
axes[1].set_title("Module 2: Discretized (5 bins)")
axes[1].set_xticks(bin_centers)
axes[1].set_xticklabels([f"Bin {i+1}\n({c:.3f})" 
                          for i, c in enumerate(bin_centers)])

plt.tight_layout()
plt.show()

Same data, different representation. The right plot has lost information — we no longer know exactly where each crab falls within its bin. But we've gained something: a clean, discrete sample space that we can reason about mathematically.


### TODO 2: Explore Different Bin Counts

How does the choice of bins change what you see? Repeat the discretization with 3 bins and 10 bins.


In [ ]:
# TODO 2: Discretize with 3 and 10 bins
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k in zip(axes, [3, 5, 10]):
    edges = np.linspace(crabs["ratio"].min(), 
                        crabs["ratio"].max(), k + 1)
    centers = [(edges[i] + edges[i+1]) / 2 for i in range(k)]
    
    # TODO: use pd.cut to assign bins, then value_counts
    bin_assign = pd.cut(crabs["ratio"], bins=edges, 
                        labels=range(1, k + 1),
                        include_lowest=True)
    bin_counts = ...  # count occurrences of each bin
    
    bw = (edges[1] - edges[0]) * 0.8
    ax.bar(centers, bin_counts.values, width=bw,
           color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_xlabel("Bin Center")
    ax.set_ylabel("Count")
    ax.set_title(f"{k} bins")

plt.tight_layout()
plt.show()

### TODO 2 (continued)

Report the count in the most-populated bin for each choice:


In [ ]:
# TODO 2: Report the mode (most populated bin count) for each
mode_3_bins = ...
mode_5_bins = ...
mode_10_bins = ...

print(f"Most populated bin count (3 bins):  {mode_3_bins}")
print(f"Most populated bin count (5 bins):  {mode_5_bins}")
print(f"Most populated bin count (10 bins): {mode_10_bins}")

## Part 3: From Counts to a PMF

A **probability mass function** (PMF) assigns a probability to each outcome in the sample space. We don't know the true PMF of our ratio bins — but we can **estimate** it from data.

The idea is simple: divide each count by the total number of crabs.

$$\hat{P}(X = k) = \frac{\text{count}_k}{n}$$

This is your **empirical PMF** — the best estimate of the true PMF given the data you have.


### TODO 3

Convert your 5-bin counts into an empirical PMF. Verify that the probabilities sum to 1.


In [ ]:
# TODO 3: Normalize counts into probabilities
n = len(crabs)
pmf = ...  # divide counts by n

print("Empirical PMF (5 bins):")
for i, (bin_label, prob) in enumerate(pmf.items()):
    print(f"  Bin {bin_label}: {prob:.4f}")

pmf_sum = ...  # sum of all probabilities
print(f"\nSum of probabilities: {pmf_sum:.4f}")
print(f"  (Should be 1.0000)")

Now let's visualize this as a proper PMF — with probabilities on the y-axis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: counts (what we had before)
axes[0].bar(bin_centers, counts.values, width=bar_width,
            color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Ratio Bin")
axes[0].set_ylabel("Count")
axes[0].set_title("Frequency (counts)")

# Right: PMF (probabilities)
axes[1].bar(bin_centers, pmf.values, width=bar_width,
            color="darkorange", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Ratio Bin")
axes[1].set_ylabel("P(X = k)")
axes[1].set_title("Empirical PMF (probabilities)")

# Add probability labels on each bar
for center, prob in zip(bin_centers, pmf.values):
    axes[1].text(center, prob + 0.005, f"{prob:.3f}",
                 ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## Part 4: Does a Simple Model Fit?

Now that we have an empirical PMF, we can ask: does a simple theoretical model match our data?

The simplest possible model: **uniform**. Every bin is equally likely.

$$P_{\text{uniform}}(X = k) = \frac{1}{5} = 0.2 \quad \text{for all } k$$

If crabs' ratios were spread evenly across the range, each bin would contain about 20% of the data. Do they?


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Empirical PMF
ax.bar(bin_centers, pmf.values, width=bar_width,
       color="darkorange", edgecolor="white", alpha=0.7,
       label="Empirical PMF")

# Uniform model
uniform_prob = 1 / n_bins
ax.axhline(y=uniform_prob, color="red", linestyle="--", 
           linewidth=2, label=f"Uniform = {uniform_prob:.2f}")

ax.set_xlabel("Ratio Bin")
ax.set_ylabel("P(X = k)")
ax.set_title("Empirical PMF vs Uniform Model")
ax.legend()

plt.tight_layout()
plt.show()

### TODO 4

Look at the plot. Does the uniform model fit well?

Which bins have more crabs than expected under the uniform model? Which have fewer? Compute the largest absolute difference between the empirical and uniform probabilities.


In [ ]:
# TODO 4: Largest absolute difference
differences = ...  # pmf.values minus uniform_prob
max_abs_diff = ...  # largest absolute difference

print("Bin-by-bin comparison:")
for i, (emp, diff) in enumerate(zip(pmf.values, differences)):
    direction = "above" if diff > 0 else "below"
    print(f"  Bin {i+1}: empirical = {emp:.4f}, "
          f"uniform = {uniform_prob:.4f}, "
          f"diff = {diff:+.4f} ({direction})")

print(f"\nLargest absolute difference: {max_abs_diff:.4f}")

## Part 5: Expectation and Variance from the PMF

Now we can use the PMF to compute summary statistics — but using the **discrete probability formulas** from lecture, not just sample averages.

For a discrete random variable $$X$$ with PMF $$p_X$$:

$$E[X] = \sum_k x_k \cdot P(X = x_k)$$

$$\text{Var}(X) = \sum_k (x_k - E[X])^2 \cdot P(X = x_k)$$

But what value do we use for $$x_k$$? Each bin represents a range of ratios. We'll use the **midpoint** (center) of each bin as the representative value. This is a natural choice: it minimizes the maximum error for any point within the bin, and it treats both edges of the bin symmetrically.


### TODO 5

Compute E[X] and Var(X) using the PMF and bin centers. Then compare to the sample mean and variance computed directly from the continuous ratio values.


In [ ]:
# The bin centers we computed earlier
print("Bin centers:", [f"{c:.4f}" for c in bin_centers])
print("PMF values: ", [f"{p:.4f}" for p in pmf.values])

# TODO 5a: Compute E[X] from the PMF
# E[X] = sum of (center * probability) for each bin
bin_centers_array = np.array(bin_centers)
pmf_array = np.array(pmf.values)

E_X_pmf = ...  # sum of center_k * P(X = k)

print(f"\nE[X] from PMF:       {E_X_pmf:.4f}")
print(f"Sample mean (continuous): {crabs['ratio'].mean():.4f}")

In [ ]:
# TODO 5b: Compute Var(X) from the PMF
# Var(X) = sum of (center_k - E[X])^2 * P(X = k)
Var_X_pmf = ...  # use E_X_pmf from above

print(f"Var(X) from PMF:          {Var_X_pmf:.6f}")
print(f"Sample variance (continuous): {crabs['ratio'].var():.6f}")

In [ ]:
# TODO 5c: How much did discretization change our estimates?
mean_diff = ...  # E_X_pmf minus sample mean
var_diff = ...   # Var_X_pmf minus sample variance

print(f"Difference in means:     {mean_diff:.4f}")
print(f"Difference in variances: {var_diff:.6f}")

**What to notice:**

The PMF-based estimates won't exactly match the sample statistics computed from the continuous data. When we binned the ratio, we replaced each exact value with a bin center — that's information loss.

The coarser the bins, the larger the discrepancy. In Module 3, we'll stop binning entirely and work with **continuous distributions** (PDFs), which avoid this problem.


## Part 6: A New Variable

So far we've treated all 100 crabs as one group — just like Module 1. But the data has been carrying a variable we haven't looked at yet.

Each crab was recorded as **Male (M)** or **Female (F)**.

Let's see if this matters.

*(And this isn't the last hidden feature in this dataset. More to come in future modules.)*


### TODO 6

How many males and females are in the dataset? Is the split balanced?


In [ ]:
# TODO 6: Count males and females
sex_counts = ...  # use value_counts() on the sex column

n_male = ...
n_female = ...

print("Sex distribution:")
print(sex_counts)
print(f"\nMales:   {n_male}")
print(f"Females: {n_female}")

## Part 7: Conditional PMFs

Here's the key Module 2 question: **does the ratio distribution differ between males and females?**

In probability language, we're asking whether the **conditional PMF** of the ratio given sex is the same for both groups:

$$\hat{P}(X = k \mid \text{Male}) \stackrel{?}{=} \hat{P}(X = k \mid \text{Female})$$

To find out, we split the data by sex and build separate PMFs.


In [ ]:
# Split by sex
males = crabs[crabs["sex"] == "M"]
females = crabs[crabs["sex"] == "F"]

print(f"Males:   {len(males)} crabs")
print(f"Females: {len(females)} crabs")

# Discretize each group using the SAME bin edges
males_binned = pd.cut(males["ratio"], bins=bin_edges,
                       labels=range(1, n_bins + 1),
                       include_lowest=True)
females_binned = pd.cut(females["ratio"], bins=bin_edges,
                         labels=range(1, n_bins + 1),
                         include_lowest=True)

# Compute conditional PMFs
pmf_male = males_binned.value_counts().sort_index() / len(males)
pmf_female = females_binned.value_counts().sort_index() / len(females)

print("\nConditional PMF (Male):")
print(pmf_male.round(4))
print("\nConditional PMF (Female):")
print(pmf_female.round(4))

### TODO 7a

Plot the two conditional PMFs side by side.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(bin_centers, pmf_male.values, width=bar_width,
            color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Ratio Bin")
axes[0].set_ylabel("P(X = k | Male)")
axes[0].set_title("Conditional PMF: Males")
axes[0].set_ylim(0, 0.5)

axes[1].bar(bin_centers, pmf_female.values, width=bar_width,
            color="coral", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Ratio Bin")
axes[1].set_ylabel("P(X = k | Female)")
axes[1].set_title("Conditional PMF: Females")
axes[1].set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

### TODO 7b

Now overlay them on the same axes for direct comparison.


In [ ]:
# TODO 7b: Overlay plot
fig, ax = plt.subplots(figsize=(9, 6))

offset = bar_width * 0.2  # slight offset so bars don't overlap
ax.bar([c - offset for c in bin_centers], pmf_male.values, 
       width=bar_width * 0.6, color="steelblue", edgecolor="white",
       alpha=0.8, label="Male")
ax.bar([c + offset for c in bin_centers], pmf_female.values, 
       width=bar_width * 0.6, color="coral", edgecolor="white",
       alpha=0.8, label="Female")

ax.set_xlabel("Ratio Bin")
ax.set_ylabel("P(X = k | Sex)")
ax.set_title("Conditional PMFs: Male vs Female")
ax.legend()

plt.tight_layout()
plt.show()

### TODO 7c

Compute the conditional means — $$E[X \mid \text{Male}]$$ and $$E[X \mid \text{Female}]$$ — using the same PMF formula from Part 5, applied to each conditional PMF.


In [ ]:
# TODO 7c: Conditional expectations
E_X_male = ...    # sum of center_k * P(X = k | Male)
E_X_female = ...  # sum of center_k * P(X = k | Female)

print(f"E[X | Male]:     {E_X_male:.4f}")
print(f"E[X | Female]:   {E_X_female:.4f}")
print(f"Difference:      {E_X_male - E_X_female:.4f}")

# Compare to direct sample means
print(f"\nSample mean (males):   {males['ratio'].mean():.4f}")
print(f"Sample mean (females): {females['ratio'].mean():.4f}")

## Part 8: The Contingency Table

So far we've looked at conditional PMFs — the distribution of ratio bins *within* each sex. A **contingency table** shows the full picture: the **joint distribution** of sex and ratio bin.

Each cell contains the count of crabs with that particular combination of sex and ratio bin.


In [ ]:
# Build the contingency table
contingency = pd.crosstab(crabs["sex"], crabs["bin"],
                           margins=True, margins_name="Total")

print("Contingency Table:")
print(contingency)

### TODO 8a

The margins (row and column totals) are the **marginal distributions**. Verify that:
- The row totals match your male/female counts from Part 6
- The column totals match your bin counts from Part 2
- The grand total is 100


In [ ]:
# TODO 8a: Verify the margins
print("Row totals (should match sex counts):")
print(f"  Males:   {contingency.loc['M', 'Total']}")
print(f"  Females: {contingency.loc['F', 'Total']}")

print("\nColumn totals (should match bin counts):")
for b in range(1, n_bins + 1):
    print(f"  Bin {b}: {contingency.loc['Total', b]}")

print(f"\nGrand total: {contingency.loc['Total', 'Total']}")

### TODO 8b

Normalize the contingency table to get the **joint empirical PMF**: $$\hat{P}(\text{Sex} = s, X = k)$$ for each combination.


In [ ]:
# TODO 8b: Normalize to get joint PMF
joint_pmf = ...  # divide the contingency table (without margins) by n

# Rebuild without margins for cleaner display
contingency_no_margins = pd.crosstab(crabs["sex"], crabs["bin"])
joint_pmf = contingency_no_margins / n

print("Joint PMF:")
print(joint_pmf.round(4))
print(f"\nSum of all entries: {joint_pmf.values.sum():.4f}")

## Part 9: Are Sex and Ratio Independent?

Two random variables $$X$$ and $$Y$$ are **independent** if and only if their joint PMF factors into the product of their marginals:

$$P(X = x, Y = y) = P(X = x) \cdot P(Y = y) \quad \text{for all } x, y$$

If sex and ratio bin are independent, then knowing a crab's sex tells you **nothing** about which ratio bin it falls in.

Let's check this visually. We'll compute what the joint PMF *would look like* if the variables were independent, and compare it to what we actually observe.


In [ ]:
# Marginal PMFs
marginal_sex = crabs["sex"].value_counts().sort_index() / n
marginal_bin = counts / n

print("Marginal PMF (sex):")
print(marginal_sex.round(4))
print("\nMarginal PMF (bin):")
print(marginal_bin.round(4))

In [ ]:
# TODO 9: Compute the independence model
# For each combination of sex and bin, multiply the marginals
independence_model = pd.DataFrame(index=["F", "M"], 
                                   columns=range(1, n_bins + 1),
                                   dtype=float)

for sex_val in ["F", "M"]:
    for bin_val in range(1, n_bins + 1):
        independence_model.loc[sex_val, bin_val] = ...  # P(sex) * P(bin)

print("If independent, joint PMF would be:")
print(independence_model.round(4))

print("\nActual observed joint PMF:")
print(joint_pmf.round(4))

In [ ]:
# Difference: observed - expected under independence
diff_table = joint_pmf - independence_model

print("Difference (observed - independent model):")
print(diff_table.round(4))

print(f"\nLargest absolute difference: "
      f"{diff_table.abs().values.max():.4f}")

### Visualize the Comparison

Let's see the observed vs independence-model joint PMFs as heatmaps.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Shared color scale
vmin = 0
vmax = max(joint_pmf.values.max(), independence_model.values.max())

# Observed
sns.heatmap(joint_pmf, annot=True, fmt=".3f", cmap="Oranges",
            vmin=vmin, vmax=vmax, ax=axes[0])
axes[0].set_title("Observed Joint PMF")
axes[0].set_ylabel("Sex")
axes[0].set_xlabel("Ratio Bin")

# Independence model
sns.heatmap(independence_model, annot=True, fmt=".3f", 
            cmap="Oranges", vmin=vmin, vmax=vmax, ax=axes[1])
axes[1].set_title("If Independent")
axes[1].set_ylabel("Sex")
axes[1].set_xlabel("Ratio Bin")

# Difference
max_diff = diff_table.abs().values.max()
sns.heatmap(diff_table, annot=True, fmt="+.3f", cmap="RdBu_r",
            vmin=-max_diff, vmax=max_diff, center=0, ax=axes[2])
axes[2].set_title("Difference\n(Observed − Independent)")
axes[2].set_ylabel("Sex")
axes[2].set_xlabel("Ratio Bin")

plt.tight_layout()
plt.show()

**How to read the difference heatmap:**

- **White cells** (near zero): the observed frequency matches what independence predicts
- **Red cells** (positive): more crabs in this cell than independence predicts
- **Blue cells** (negative): fewer crabs than independence predicts

If sex and ratio were truly independent, all cells would be white. Are they?

We don't yet have the tools to say whether these differences are **statistically significant** or just due to random variation in a small sample. That's coming in Module 5. For now, notice the pattern — and carry the question with you.


## Part 10: Looking Back, Looking Forward

Let's put one final plot together. Your original Module 1 question was "one population or two?" — now you can see it through the lens of sex.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the overall distribution (Module 1 view)
sns.histplot(crabs["ratio"], bins=15, kde=True, color="seagreen",
             edgecolor="white", alpha=0.7, linewidth=0.5,
             line_kws={"linewidth": 2.5}, ax=axes[0])
axes[0].set_xlabel("FL / CL Ratio")
axes[0].set_ylabel("Count")
axes[0].set_title("All Crabs (Module 1 View)")

# Right: split by sex (Module 2 view)
sns.histplot(data=crabs, x="ratio", hue="sex", bins=15, 
             kde=True, alpha=0.5, edgecolor="white",
             linewidth=0.5, line_kws={"linewidth": 2.5},
             palette={"M": "steelblue", "F": "coral"},
             ax=axes[1])
axes[1].set_xlabel("FL / CL Ratio")
axes[1].set_ylabel("Count")
axes[1].set_title("Split by Sex (Module 2 View)")

plt.tight_layout()
plt.show()

### What You Can See Now That You Couldn't Before

In Module 1, you had one histogram and a vague sense that something might be going on.

Now you have:
- A **discrete probability model** (the empirical PMF)
- A **theoretical baseline** to compare against (uniform)
- A way to **quantify** summaries (E[X], Var(X))
- **Conditional distributions** that reveal group differences
- A **joint distribution** that encodes the full picture
- A **framework for independence** to test whether groups matter

You also know that the uniform model doesn't fit well, and that males and females might have different ratio distributions. But you can't yet say *how confident* you are in these conclusions.

**What's coming:**
- **Module 3**: Stop binning. Work with the continuous ratio directly. Add more measurements per crab. Model multivariate relationships.
- **Module 4**: Estimate parameters, quantify uncertainty, and make decisions.
- **Module 5**: Formally test whether sex matters, whether the distribution is bimodal, and whether a model fits.

And there's still more to learn about these crabs.


## Report Your Results

Enter these values on the course platform. Round as indicated.


In [ ]:
print("="*55)
print("MODULE 2 REPORT VALUES")
print("="*55)
print(f"R1.  Mean ratio (4 decimals):          "
      f"{crabs['ratio'].mean():.4f}")
print(f"R2.  Mode count, 3 bins:               "
      f"{mode_3_bins}")
print(f"R3.  Mode count, 5 bins:               "
      f"{mode_5_bins}")
print(f"R4.  Mode count, 10 bins:              "
      f"{mode_10_bins}")
print(f"R5.  PMF: P(most populated bin):        "
      f"{pmf.max():.4f}")
print(f"R6.  Max |empirical - uniform|:         "
      f"{max_abs_diff:.4f}")
print(f"R7.  E[X] from PMF (4 decimals):       "
      f"{E_X_pmf:.4f}")
print(f"R8.  E[X]_PMF - sample mean:           "
      f"{mean_diff:.4f}")
print(f"R9.  Number of males:                   "
      f"{n_male}")
print(f"R10. E[X | Male] (4 decimals):         "
      f"{E_X_male:.4f}")
print(f"R11. E[X | Female] (4 decimals):       "
      f"{E_X_female:.4f}")
print(f"R12. Largest |observed - independent|:  "
      f"{diff_table.abs().values.max():.4f}")

## Reflection Questions

**Q1.** In Part 2, you saw the same data discretized into 3, 5, and 10 bins. As a modeler, what are you trading off when you choose more bins? What do you gain? What do you lose?


*Your answer here*

**Q2.** In Part 5, your PMF-based E[X] didn't exactly match the sample mean. Which estimate do you think is "better"? Why? Under what circumstances would the PMF-based estimate be preferred?


*Your answer here*

**Q3.** Look at your conditional PMFs from Part 7. Describe the differences between males and females in plain language. If you had to bet whether sex affects the ratio, which way would you bet right now — and how confident are you?


*Your answer here*

**Q4.** The independence heatmap in Part 9 showed differences between the observed and independent-model joint PMFs. But with only 100 crabs split across 10 cells, some differences are expected even if the variables ARE independent. How could you tell the difference between "real dependence" and "random noise"? (You don't need a formal answer — just your intuition.)


*Your answer here*

**Q5.** We hinted that there's more hidden structure in this dataset. Given what you've seen so far — the spread in the ratio, the possible differences between sexes, the shape of the distribution — what do you think we might reveal next? What would explain the patterns you're seeing?


*Your answer here*